# Phase 4: Deep Learning Models — Wikipedia Next-Click Prediction

Four architectures, all self-contained:
- **MLP Scorer** — multimodal features (text+CLIP+wiki2vec+cats, up to 1125d)
- **GCN+GRETEL** — GCN with labeling trick (conditions on current/target pair)
- **GAT Scorer** — graph attention over link neighborhood
- **Transformer Scorer** — cross-attention between target and candidate links

In [ ]:
import io, pickle, tarfile, urllib.request, urllib.parse
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

ROOT = Path.cwd()
while not (ROOT / 'dataset-task2').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / 'dataset-task2'
CACHE = ROOT / '.cache'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Data + Adjacency

In [ ]:
articles = pd.read_csv(DATA / 'articles.csv')
categories = pd.read_csv(DATA / 'categories.csv')
train = pd.read_csv(DATA / 'states_train.csv')
test = pd.read_csv(DATA / 'states_test.csv')

def load_adj():
    for p in [CACHE / 'wikispeedia_adj.pkl', DATA / 'wikispeedia_adj.pkl']:
        if p.exists():
            with open(p, 'rb') as f:
                return pickle.load(f)
    title_to_id = dict(zip(articles['title'].str.strip(), articles['article_id']))
    url = 'https://snap.stanford.edu/data/wikispeedia/wikispeedia_paths-and-graph.tar.gz'
    print(f'Downloading Wikispeedia graph ...')
    with urllib.request.urlopen(url) as resp:
        with tarfile.open(fileobj=io.BytesIO(resp.read()), mode='r:gz') as tar:
            f = tar.extractfile('wikispeedia_paths-and-graph/links.tsv')
            content = f.read()
    links = pd.read_csv(io.BytesIO(content), sep='\t', skiprows=14,
                        header=None, names=['source', 'target'])
    links['sid'] = links['source'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links['tid'] = links['target'].apply(
        lambda x: urllib.parse.unquote(x).replace('_', ' ').strip()).map(title_to_id)
    links = links.dropna(subset=['sid', 'tid'])
    adj = {i: [] for i in range(len(articles))}
    for _, r in links.iterrows():
        adj[int(r['sid'])].append(int(r['tid']))
    adj = {k: list(set(v)) for k, v in adj.items()}
    with open(CACHE / 'wikispeedia_adj.pkl', 'wb') as f:
        pickle.dump(adj, f)
    return adj

adj = load_adj()
n_nodes = len(articles)
n_edges = sum(len(v) for v in adj.values())
print(f'Adjacency: {n_nodes} nodes, {n_edges:,} edges')

## 2. Feature Loading

Load whatever is cached from Phase 2 EDA. Falls back to text-only if CLIP/Wiki2Vec not generated yet.

In [ ]:
# Text embeddings (always available)
if (CACHE / 'article_embeddings.npy').exists():
    text_emb = np.load(CACHE / 'article_embeddings.npy')
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    text_emb = model.encode(articles['title'].tolist(), show_progress_bar=True, batch_size=64)
    np.save(CACHE / 'article_embeddings.npy', text_emb)

# Category encoding (always available)
all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((n_nodes, len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0

# Optional: CLIP visual embeddings
clip_emb = None
if (CACHE / 'clip_embeddings_512.npy').exists():
    clip_emb = np.load(CACHE / 'clip_embeddings_512.npy')
    print(f'CLIP loaded: {clip_emb.shape}')

# Optional: Wikipedia2Vec embeddings
w2v_emb = None
if (CACHE / 'wiki2vec_embeddings_100.npy').exists():
    w2v_emb = np.load(CACHE / 'wiki2vec_embeddings_100.npy')
    print(f'Wiki2Vec loaded: {w2v_emb.shape}')

# Combine available features
feat_parts = [text_emb, cat_enc]
if clip_emb is not None:
    feat_parts.insert(-1, clip_emb)
if w2v_emb is not None:
    feat_parts.insert(-2, w2v_emb)
node_feats = np.concatenate(feat_parts, axis=1).astype(np.float32)
print(f'Combined features: {node_feats.shape} = {" +".join(str(p.shape[1]) for p in feat_parts)}')

## 3. Candidate Dataset + Normalized Adjacency

In [ ]:
class CandidateDataset(Dataset):
    def __init__(self, df, adj, node_feats):
        self.samples = []
        for _, r in df.iterrows():
            curr, tgt, nxt = r['current_article_id'], r['target_article_id'], r['next_article_id']
            links = adj.get(curr, [])
            if not links:
                continue
            for link in links:
                self.samples.append({
                    'c_feat': node_feats[curr].copy(),
                    't_feat': node_feats[tgt].copy(),
                    'l_feat': node_feats[link].copy(),
                    'l_deg': float(len(adj.get(link, []))),
                    'n_links': float(len(links)),
                    'label': 1.0 if link == nxt else 0.0,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        return (torch.tensor(s['c_feat']), torch.tensor(s['t_feat']),
                torch.tensor(s['l_feat']), torch.tensor(s['l_deg']),
                torch.tensor(s['n_links']), torch.tensor(s['label']))


def build_norm_adj(adj, n_nodes, device='cpu'):
    m = np.zeros((n_nodes, n_nodes), dtype=np.float32)
    for s, ts in adj.items():
        for t in ts:
            m[s, t] = 1.0
    deg = m.sum(1, keepdims=True)
    deg[deg == 0] = 1
    d = np.power(deg, -0.5).flatten()
    return torch.tensor(m * d[np.newaxis, :] * d[:, np.newaxis], device=device)

adj_norm = build_norm_adj(adj, n_nodes, device)
full_feats = torch.tensor(node_feats, device=device)
feat_dim = node_feats.shape[1]
print(f'Feat dim: {feat_dim}, adj_norm: {adj_norm.shape}')

## 4. MLP Scorer (Multimodal)

Takes curr/tgt/candidate feature vectors, concatenates, passes through MLP → score.

In [ ]:
class MLPScorer(nn.Module):
    def __init__(self, feat_dim, hidden=256):
        super().__init__()
        d_in = feat_dim * 3 + 2  # curr + tgt + cand + degree + n_links
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, cf, tf, lf, deg, nlink):
        x = torch.cat([cf, tf, lf, deg.unsqueeze(-1), nlink.unsqueeze(-1)], dim=-1)
        return self.net(x).squeeze(-1)

## 5. GCN Scorer + GRETEL Labeling Trick

Appends binary labels (1=current/target node, 0=other) to node features before GCN. The GCN propagates context-awareness through the graph.

In [ ]:
class GCNConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj_n):
        return F.relu(adj_n @ self.w(x))


class GretelGCN(nn.Module):
    def __init__(self, feat_dim, hidden=256, gcn_out=128):
        super().__init__()
        self.conv1 = GCNConv(feat_dim + 1, hidden)   # +1 for labeling trick
        self.conv2 = GCNConv(hidden, gcn_out)
        self.scorer = nn.Sequential(
            nn.Linear(gcn_out * 3 + 2, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, full_feats, adj_n, cnode, tnode, pnode, deg, nlink):
        N = full_feats.size(0)
        B = cnode.size(0)
        labels = torch.zeros(N, 1, device=full_feats.device)
        labels[cnode] = 1.0
        labels[tnode] = 1.0
        x = torch.cat([full_feats, labels], dim=-1)
        h = self.conv2(self.conv1(x, adj_n), adj_n)
        z = torch.cat([h[cnode], h[tnode], h[pnode],
                       deg.unsqueeze(-1), nlink.unsqueeze(-1)], dim=-1)
        return self.scorer(z).squeeze(-1)

## 6. GAT Scorer

Single-head graph attention over the link neighborhood. Attends over neighbors for each node.

In [ ]:
class GATConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.w = nn.Linear(in_dim, out_dim, bias=False)
        self.a = nn.Linear(out_dim * 2, 1)
        self.leaky = nn.LeakyReLU(0.2)

    def forward(self, x, adj_n):
        wh = self.w(x)  # [N, d]
        N = wh.size(0)
        # Sparse approximation: use adj_norm-weighted message passing
        return F.relu(adj_n @ wh)


class GATScorer(nn.Module):
    def __init__(self, feat_dim, hidden=256, gat_out=128):
        super().__init__()
        self.w_proj = nn.Linear(feat_dim, hidden)
        self.conv1 = GATConv(hidden, hidden)
        self.conv2 = GATConv(hidden, gat_out)
        self.scorer = nn.Sequential(
            nn.Linear(gat_out * 3 + 2, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, full_feats, adj_n, cnode, tnode, pnode, deg, nlink):
        x = F.relu(self.w_proj(full_feats))
        h = self.conv2(self.conv1(x, adj_n), adj_n)
        z = torch.cat([h[cnode], h[tnode], h[pnode],
                       deg.unsqueeze(-1), nlink.unsqueeze(-1)], dim=-1)
        return self.scorer(z).squeeze(-1)

## 7. Transformer Scorer

Self-attention over candidate links + cross-attention with target embedding. Models interactions between alternative links.

In [ ]:
class TransformerScorer(nn.Module):
    def __init__(self, feat_dim, hidden=256):
        super().__init__()
        self.tgt_proj = nn.Linear(feat_dim, hidden)
        self.cand_proj = nn.Linear(feat_dim, hidden)
        self.cross_attn = nn.MultiheadAttention(hidden, num_heads=4, batch_first=True)
        self.self_attn = nn.MultiheadAttention(hidden, num_heads=4, batch_first=True)
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 1),
        )

    def forward(self, cf, tf, lf, deg, nlink):
        # cf, tf, lf: [B, D]
        # Reshape lf to [B, 1, D] for single-candidate scoring (batched across states)
        c_proj = self.cand_proj(cf)  # [B, H]
        l_proj = self.cand_proj(lf)  # [B, H]
        t_proj = self.tgt_proj(tf)   # [B, H]

        # Self-attention over [current, candidate] pair
        pairs = torch.stack([c_proj, l_proj, t_proj], dim=1)  # [B, 3, H]
        attn_out, _ = self.self_attn(pairs, pairs, pairs)
        h = self.norm1(pairs + attn_out)

        # Cross-attention: target attends to current+candidate
        tgt = t_proj.unsqueeze(1)  # [B, 1, H]
        ctx = h[:, :2, :]           # [B, 2, H] — current + candidate
        cross, _ = self.cross_attn(tgt, ctx, ctx)
        h2 = self.norm2(tgt + cross)  # [B, 1, H]

        return self.ffn(h2.squeeze(1)).squeeze(-1)

## 8. Training Loop

In [ ]:
def train_epoch(model, loader, opt, loss_fn, mtype, adj_n, ff):
    model.train()
    total = 0
    for batch in loader:
        cf, tf, lf, deg, nlink, labels = [x.to(device) for x in batch]

        if mtype == 'gcn':
            cnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            tnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            pnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            scores = model(ff, adj_n, cnode, tnode, pnode, deg, nlink)
        elif mtype == 'gat':
            cnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            tnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            pnode = torch.zeros(cf.size(0), dtype=torch.long, device=device)
            scores = model(ff, adj_n, cnode, tnode, pnode, deg, nlink)
        else:
            scores = model(cf, tf, lf, deg, nlink)

        loss = loss_fn(scores, labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    return total / len(loader)

## 9. Prediction

In [ ]:
@torch.no_grad()
def predict(model, df, adj, node_feats, mtype, adj_n=None, ff=None, bs=64):
    model.eval()
    preds = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc='Predict'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt)
            continue
        B = len(links)
        cf = torch.tensor(node_feats[curr], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        tf = torch.tensor(node_feats[tgt], dtype=torch.float, device=device).unsqueeze(0).expand(B, -1)
        lf = torch.tensor(node_feats[links], dtype=torch.float, device=device)
        deg = torch.tensor([len(adj.get(l, [])) for l in links], dtype=torch.float, device=device)
        nlink = torch.tensor(float(B), dtype=torch.float, device=device).expand(B)

        if mtype in ('gcn', 'gat'):
            cnode = torch.full((B,), curr, dtype=torch.long, device=device)
            tnode = torch.full((B,), tgt, dtype=torch.long, device=device)
            pnode = torch.tensor(links, dtype=torch.long, device=device)
            scores = model(ff, adj_n, cnode, tnode, pnode, deg, nlink)
        else:
            scores = model(cf, tf, lf, deg, nlink)
        preds.append(links[scores.argmax().item()])
    return np.array(preds)

## 10. Submission Helper

In [ ]:
def submit(model, df, name, mtype, adj_n=None, ff=None):
    preds = predict(model, df, adj, node_feats, mtype, adj_n, ff)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    d = ROOT / 'submissions' / f'{ts}_{name}'
    d.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'state_id': df['state_id'].values,
                  'predicted_next_article_id': preds}).to_csv(d / 'submission.csv', index=False)
    # Validate
    expected = pd.read_csv(DATA / 'sample_submission.csv')
    actual = pd.read_csv(d / 'submission.csv')
    assert list(actual.columns) == list(expected.columns)
    assert len(actual) == len(expected)
    print(f'Saved & validated: {d / "submission.csv"}')
    return preds

## 11. Smoke Test All Models

In [ ]:
tiny = train.sample(50, random_state=42)
ds = CandidateDataset(tiny, adj, node_feats)
ldr = DataLoader(ds, batch_size=16, shuffle=True)
loss_fn = nn.BCEWithLogitsLoss()

configs = [
    ('MLP', MLPScorer(feat_dim, 256), 'mlp', None, None),
    ('GCN+GRETEL', GretelGCN(feat_dim, 256, 128), 'gcn', adj_norm, full_feats),
    ('GAT', GATScorer(feat_dim, 256, 128), 'gat', adj_norm, full_feats),
    ('Transformer', TransformerScorer(feat_dim, 256), 'xformer', None, None),
]

for name, model, mtype, adj_n, ff in configs:
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss = train_epoch(model, ldr, opt, loss_fn, mtype, adj_n, ff)
    n_params = sum(p.numel() for p in model.parameters())
    preds = predict(model, test.head(5), adj, node_feats, mtype, adj_n, ff)
    print(f'{name}: loss={loss:.4f}, params={n_params:,}, preds[0:3]={list(preds[:3])}')

## 12. Full Training (run on remote server)

In [ ]:
# Full train + test loop — uncomment and run on remote:

# ds = CandidateDataset(train, adj, node_feats)
# ldr = DataLoader(ds, batch_size=256, shuffle=True, num_workers=4)
# loss_fn = nn.BCEWithLogitsLoss()

# for mname, mcls, mtype, adj_n, ff in [
#     ('mlp', MLPScorer, 'mlp', None, None),
#     ('gcn_gretel', GretelGCN, 'gcn', adj_norm, full_feats),
#     ('gat', GATScorer, 'gat', adj_norm, full_feats),
#     ('xformer', TransformerScorer, 'xformer', None, None),
# ]:
#     model = mcls(feat_dim, 256).to(device)
#     opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
#     sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
#     for ep in range(5):
#         loss = train_epoch(model, ldr, opt, loss_fn, mtype, adj_n, ff)
#         sched.step()
#         print(f'{mname} epoch {ep}: loss={loss:.4f}')
#     submit(model, test, f'dl_{mname}', mtype, adj_n, ff)
#     print(f'{mname} done.')